# M1.5 · M1.5_本征值与模态 — 交互式实验 Notebook

> 本 Notebook 按对应 HTML 演示页逐页整理；页面文字、公式和可运行代码均来自该页。
> 可运行代码直接包含在 Notebook 中，不查找、不导入当前课程项目里的文件。
> 运行环境：Python 3、NumPy、SciPy、Matplotlib；在 Jupyter 中按顺序执行代码单元。

## 演示页目录

1. M1.5 本征值与模态
2. ① 工程场景：敲一下鼓，为什么只在一个频率上“响”
3. ② 失效先行：数值第二阶和第三阶“对不上”教科书
4. ③ 物理直觉：先把“响应”拆成“允许的振动方式”
5. ④ 从波动方程到矩形膜解析谱
6. ⑤ 正交性、归一化与模态展开
7. ⑥ 正方形的简并：频率相同不代表模态相同
8. ⑦ 最小代码：解析谱与离散谱对照
9. ⑧ 验证三件套：谱、残差、正交性
10. ⑨ POE：改变几何和边界，先预测谱如何移动
11. ⑩ 工程桥接：广义本征问题不是黑盒
12. ⑪ 现场诊断挑战（代码面试）：看证据，找病因
13. ⑫ 本节验收清单
14. 总结与下一站

## 第 1 页 · M1.5 本征值与模态

主讲 · 工程物理场 AI 科学计算课程组 所属分层 · T1 · 进阶主线（压缩课时可选） 核心方程 · $-\nabla^2\phi=\lambda\phi$ 、 $Kq=\omega^2Mq$ 核心任务 · 用矩形膜解析谱验证本征求解器，理解正交模态、简并和工程广义本征问题。 从“一个解”升级到“系统会怎样振动” 先修要求：M1.3 分离变量、M1.4 波动方程；会读内积、矩阵特征值和基本 Python 数组。 T1 内容强调诊断和谱验证；若课程压缩到 32 学时，可保留矩形膜基准而略去拓展推导。

## 第 2 页 · ① 工程场景：敲一下鼓，为什么只在一个频率上“响”

鼓膜、矩形薄板或水面振动会出现稳定的节点线和腹部。改变激励频率，系统不是任意响应，而是在靠近某些固有频率时放大。 30 秒课堂思考：把鼓膜的边界从“固定在框上”改成“自由滑动”，基频应该变高、变低，还是不变？先写下判断，再运行第 ⑦ 页看离散谱如何移动。（提示：约束减弱 ⇒ 刚度下降 ⇒ 频率下降，而且自由边界会多出一个 $\lambda\approx0$ 的刚体平移模。）

## 第 3 页 · ② 失效先行：数值第二阶和第三阶“对不上”教科书

真实评审现场：正方形膜的模态测试评审——数值第二、第三阶“模态云图”与教科书解析振型逐点不同，评审据此判定模型有错、要求返工。但特征值与前两阶子空间投影全部正确。 现象：本征残差全程 $10^{-13}$ 不变，逐点匹配度却在 $0.707\ldots1.000$ 之间摆动。 先写下你的判断：这是求解器没收敛（A）、网格太粗（B），还是根本不该逐点比振型（C）？第 ⑪ 页揭晓。 运行前要观察的量：最大本征残差 $\lVert Kq-\lambda Mq\rVert/\lVert Kq\rVert$ 、逐点 $\lvert\cos\rvert$ 、以及子空间投影残差 $\lVert VV^{\!T}MQ-Q\rVert/\lVert Q\rVert$ 。

## 第 4 页 · ③ 物理直觉：先把“响应”拆成“允许的振动方式”

先不碰公式，先想三件事—— 1. 弦上只能跑“整数个半波”。拨一根两端固定的弦，它只能以 $L$ 、 $2L$ /2、 $3L$ /3…… 为半波长振动。把弦两端固定，位移必须为零，这就把“可能的形状”限死成一族离散模式——就像只允许整数阶的台阶，不允许斜坡。 2. 每个模式自己有一个“固有频率”。你敲一下弦，能量不会均摊到所有模式上，而是优先灌进那个特征长度恰好匹配的模式。换个激励频率，就换个模式被点亮——这就是共振。 3. 模式之间“各走各的”。两个模式叠加时，它们在每一点都同时存在，谁也盖不住谁。正交性（内积为零）说的就是：两个模式在同一时刻的“影子”不重叠。把这一点想清楚了，“能量按模态分解”才有意义——因为总能量可以拆成互不干扰的几份。 所以：特征值回答“系统允许哪些频率”，特征向量回答“能量以什么形状组织”。两者必须分别验证——这是本节所有失效的根源。 网格纸类比（为什么高频越来越难算准）：一个模式在纸上画 $m$ 个半波，每半波只分到几个格子时，格子数 $\|m$ 就得跟着涨。 $m$ 越大，格子不够分，形状就画歪。“高阶模态不准”往往不是求解器不行，而是网格分辨率不够——这正是第 ⑦ 页要报告收敛阶的原因。

## 第 5 页 · ④ 从波动方程到矩形膜解析谱

令 $u(x,y,t)=\phi(x,y)\cos(\omega t)$ ，代入二维波动方程得到 Helmholtz 本征问题 $$-\nabla^2\phi=\lambda\phi,\qquad \lambda=\omega^2/c^2.$$ 空间边界条件决定允许的 $\lambda$ ；时间因子只负责以 $\omega=c\sqrt\lambda$ 振荡。这个分离解释了“模态形状”和“固有频率”为什么可以分开验证。 对 $0 $$\phi_{mn}(x,y)=\sin\frac{m\pi x}{a}\sin\frac{n\pi y}{b},$$ $$\lambda_{mn}=\pi^2\left[\left(\frac{m}{a}\right)^2+\left(\frac{n}{b}\right)^2\right],\quad \omega_{mn}=c\sqrt{\lambda_{mn}}.$$ 第一批模态 $(1,1),(1,2),(2,1),(2,2)$ 就构成二维本征求解器的最小回归测试。

## 第 6 页 · ⑤ 正交性、归一化与模态展开

在均匀矩形区域， $$\int_0^a\int_0^b\phi_{mn}\phi_{pq}\,dxdy=0\quad\text{当 }(m,n)\ne(p,q).$$ 任意初始位移可展开为 $$u(x,y,0)=\sum_{m,n}A_{mn}\phi_{mn}(x,y).$$ 离散后对应矩阵内积 $q_i^TMq_j=\delta_{ij}$ 。因此模态比较必须说明使用的是欧氏归一化还是质量矩阵归一化。

## 第 7 页 · ⑥ 正方形的简并：频率相同不代表模态相同

当 $a=b$ 时， $(m,n)=(1,2)$ 与 $(2,1)$ 有相同的 $\lambda$ ，形成简并子空间。求解器返回的两个向量可以是这两个解析模态的任意正交线性组合。 诊断规则：简并时不要逐向量比较符号和形状；应比较特征值、子空间投影和模态正交性。矩形稍微拉长后简并解除，正是一个很好的 POE 实验。

## 第 8 页 · ⑦ 最小代码：解析谱与离散谱对照

点击左侧 ▶ 运行 后，文本输出与图像会显示在这里 本机实测输出（Python 3.12 / numpy 2.4.6 / scipy 1.17.1；Q1 双线性单元 + 一致质量矩阵，Dirichlet 消元）： text 理论前 9 阶: [(1, 1, 19.739209), (1, 2, 49.348022), (2, 1, 49.348022), (2, 2, 78.956835), (1, 3, 98.696044), (3, 1, 98.696044), (2, 3, 128.304857), (3, 2, 128.304857), (3, 3, 177.652879)] 网格 λ₁ 数值 λ₁ 解析 r₁ 观测阶 8x10 16.907527 16.723496 1.100e-02 - 16x20 16.769350 16.723496 2.742e-03 2.00 32x40 16.734950 16.723496 6.849e-04 2.00 M-正交性 n=24 前 6 阶 max|Phi^T M Phi - I| = 8.882e-16 前 6 阶本征值: [19.76741 49.588118 49.588118 79.408826 99.857431 99.857431] 相对误差 $$r_i=\frac{|\lambda_i^{ ext{num}}-\lambda_i^{ ext{exact}}|}{|\lambda_i^{ ext{exact}}|}$$ 每 2 倍网格降约 4 倍，观测阶 2.00 / 2.00 ⇒ 二阶格式得到证实。但收敛阶只能证明格式阶数，证明不了矩阵装对了——装配错误同样会二阶收敛。模态残差 $\lVert Kq_i-\lambda_iMq_i\rVert$ 必须单独看（见下页）。

In [ ]:
import numpy as np
from scipy.linalg import eigh

def rectangular_spectrum(a=1.0, b=1.0, m_max=3, n_max=3):
    """解析谱：λ_mn = π²[(m/a)² + (n/b)²]"""
    return sorted((np.pi**2 * ((m/a)**2 + (n/b)**2), m, n)
                  for m in range(1, m_max + 1) for n in range(1, n_max + 1))

def analytic_lambda(m, n, a=1.0, b=1.0):
    return np.pi**2 * ((m/a)**2 + (n/b)**2)

# 离散侧：Q1 双线性矩形单元（2×2 高斯积分 + 一致质量矩阵 + 四边 Dirichlet 消元）。
# 与 tests/test_membrane_spectrum.py 同一离散口径；浏览器里没有本地文件系统，
# 所以这里自带一份，数值与本机实测逐位一致（同一 LAPACK 广义本征，差异 ~1e-15）。
def _q1_km(nx, ny, a=1.0, b=1.0):
    hx, hy = a/nx, b/ny
    idx = lambda i, j: j*(nx+1) + i
    nn = (nx+1)*(ny+1)
    K = np.zeros((nn, nn)); M = np.zeros((nn, nn))
    detJ, g = hx*hy/4.0, 1.0/np.sqrt(3.0)
    for xi, eta in ((-g, -g), (g, -g), (g, g), (-g, g)):
        N = np.array([(1-xi)*(1-eta), (1+xi)*(1-eta), (1+xi)*(1+eta), (1-xi)*(1+eta)])/4.0
        dNx = np.array([-(1-eta), (1-eta), (1+eta), -(1+eta)])/4.0 * 2.0/hx
        dNy = np.array([-(1-xi), -(1+xi), (1+xi), (1-xi)])/4.0 * 2.0/hy
        Ke = (np.outer(dNx, dNx) + np.outer(dNy, dNy))*detJ
        Me = np.outer(N, N)*detJ
        for ex in range(nx):
            for ey in range(ny):
                e = [idx(ex, ey), idx(ex+1, ey), idx(ex+1, ey+1), idx(ex, ey+1)]
                K[np.ix_(e, e)] += Ke; M[np.ix_(e, e)] += Me
    inner = [idx(i, j) for j in range(1, ny) for i in range(1, nx)]
    return K[np.ix_(inner, inner)], M[np.ix_(inner, inner)]

def solve_spectrum(nx, ny, a=1.0, b=1.0, k=1):
    K, M = _q1_km(nx, ny, a, b)
    vals, vecs = eigh(K, M)                 # 对称正定束 Kφ = λMφ
    return vals[:k], vecs[:, :k], K, M

print("理论前 9 阶:", [(m, n, round(lam, 6)) for lam, m, n in rectangular_spectrum()])
lam_exact = analytic_lambda(1, 1, 1.0, 1.2)
print(f"{'网格':>8}{'λ₁ 数值':>14}{'λ₁ 解析':>14}{'r₁':>12}{'观测阶':>8}")
prev = None
for nx, ny in ((8, 10), (16, 20), (32, 40)):
    vals, _, _, _ = solve_spectrum(nx, ny, a=1.0, b=1.2, k=1)
    r = abs(vals[0] - lam_exact)/lam_exact
    order = "-" if prev is None else f"{np.log2(prev/r):.2f}"
    print(f"{f'{nx}x{ny}':>8}{vals[0]:>14.6f}{lam_exact:>14.6f}{r:>12.3e}{order:>8}")
    prev = r

# 收敛阶只能证明格式阶数，证明不了矩阵装对了 —— M-正交性必须单独看。
vals, vecs, K, M = solve_spectrum(24, 24, a=1.0, b=1.0, k=6)
G = vecs.T @ M @ vecs
dev = np.abs(G - np.eye(len(vals))).max()
print(f"M-正交性 n=24 前 6 阶 max|Phi^T M Phi - I| = {dev:.3e}")
print("  前 6 阶本征值:", np.array2string(vals, precision=6))
  

## 第 9 页 · ⑧ 验证三件套：谱、残差、正交性

检查 通过标准 本机实测（python tests/test_membrane_spectrum.py --dump） 失败提示 解析谱误差网格加密后前几阶 $r_i$ 下降 $1.100 imes10^{-2} o2.742 imes10^{-3} o6.849 imes10^{-4}$ ，观测阶 2.00 / 2.00边界或单位错误 本征残差 $\lVert Kq_i-\lambda_iMq_i\rVert$ 足够小 $n=24$ 时前四阶均 $\approx10^{-13}$ 求解器未收敛/矩阵装配错 模态正交性 $Q^TMQ\approx I$ $M$ -归一化：与 $I$ 最大偏差 8.88e-16质量矩阵或归一化错误 能量分解模态能量与总能量一致 $M$ -归一化： $\Sigma c_i^2=3.46947612$ ，偏差 0.000e+00漏模态或重复计数 本节最大的一个坑（实测数据）：若改用欧氏归一化（ $\lVert q\rVert_2=1$ 而非 $q^TMq=1$ ）， $Q^TMQ$ 的非对角项仍只有 $2.80 imes10^{-19}$ （依然正交！），坏的是对角： $D=[1.73,1.71,1.71,1.05,0.70,0.20] imes10^{-3}$ ，极差比 **8.798**。更隐蔽的是——只抽查前六阶时极差比只有 1.023，看起来完全正常；必须跨整个谱抽样才暴露。能量分解则相差 **594 倍**。 不要用“特征值列表看起来像”替代残差和正交性证据。运行结果以本机实测为准。

## 第 10 页 · ⑨ POE：改变几何和边界，先预测谱如何移动

💡 POE 自检台：先独立思考（可点击填写预测），再点击右侧按钮揭晓标准解析对照。 改动 运行前预测 观测证据 解释与失效归类 自检操作 $a$ 增大 点击先猜... [待揭晓][标准预测] $x$ 向模态频率下降 [待揭晓] $\lambda_{m n}$ 的解析变化 [待揭晓]尺度效应，非缺陷：频率随特征长度反比下降 正方形变成长方形 点击先猜... [待揭晓][标准预测] 简并解除 [待揭晓] $(1,2)$ 、 $(2,1)$ 分裂 [待揭晓]对称性破缺，非失效；误判为装配错属 F4 固定边界改为自由边界 点击先猜... [待揭晓][标准预测] 低阶频率下降 [待揭晓]边界条件下的谱对照 [待揭晓]约束减弱使刚度下降；若频率反升说明边界装配反了（F4） 网格加密 点击先猜... [待揭晓][标准预测] 前几阶更接近精确谱 [待揭晓]相对误差收敛曲线 [待揭晓]离散收敛证据；若不收敛则查边界或矩阵装配（F4）

## 第 11 页 · ⑩ 工程桥接：广义本征问题不是黑盒

有限元离散后常见形式为 $$Kq=\omega^2Mq.$$ $K$ 是刚度/扩散算子， $M$ 是质量/惯性度量； 边界条件改变 $K$ 、 $M$ 的自由度和零空间； 解析矩形膜是验证工业模态分析的低成本基准。 M3 将进一步比较不同网格、材料和边界下的谱；本节先确保“解出来的频率”能追溯到方程与模态。

## 第 12 页 · ⑪ 现场诊断挑战（代码面试）：看证据，找病因

真实工程现场：正方形膜数值第二、三阶模态云图旋转 45° 导致评审要求返工；矩形膜模态云图接近解析解，但质量矩阵正交性检查 Qᵀ M Q 却严重偏差。请审查以下两组真实运行证据： 点选答案立即出解析，先看证据再找病因。全部答完给出诊断得分与排查建议。

## 第 13 页 · ⑫ 本节验收清单

能从二维波动方程推导 Helmholtz 本征问题； 能写出矩形膜前四阶解析本征值和模态； 能解释正方形简并及其子空间比较方法； 能报告谱误差、模态残差和 $M$ -正交性； pytest tests -q 全绿（14 passed）； 说明该节在 T1 中的可选压缩范围。 练习阶梯（五级）： 跑通：python code/solution.py，核对终端的离散谱→解析谱收敛表（三档误差 + 观测阶）。 改参数并解释：把矩形拉长（ $b/a$ = 1.00 → 1.02 → 1.20），观察简并间距 $|\lambda_3-\lambda_2|/\lambda_2$ 从 $2.72\times10^{-14}$ 跳到 $2.41\times10^{-2}$ 、 $2.43\times10^{-1}$ 。 预测：在 notebooks/episode_lab.ipynb 预测格写下“固定边界改自由边界后基频如何变、会不会多出零频模”，再执行验证。 诊断：完成第 ⑪ 页两道 ABCD 根因排查，在 diagnostics/F4_简并与归一化口径.md 核对现象、根因与修复。 迁移：把“比较子空间而非逐点比较”的判据用到非均匀密度的广义本征问题 $Kq=\omega^2Mq$ 。 分层练习：T1 验证解析谱；T2 推导非均匀密度下的广义本征问题。

## 第 14 页 · 总结与下一站

本征值给出系统的允许频率，模态给出能量如何组织。下一节进入 M1.6 柱/球坐标、特殊函数与声辐射，处理曲面几何、轴心正则性以及圆柱/球体的出射波基准。